In [ ]:
# ==================== Swin Transformer 多标签分类训练（极端过采样版）====================

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import gc
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import timm
import warnings
warnings.filterwarnings('ignore')

# 导入你的模块
import sys
sys.path.append('..')

try:
    from src.data.dataset import FundusDataset
    from src.data.focal_loss import FocalLoss
    from src.data.config import config, device
    print("✅ 成功导入自定义模块")
    print(f"使用设备: {device}")
except Exception as e:
    print(f"❌ 导入模块失败: {e}")
    raise

# ========== 数据路径 ==========
train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation images'
excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'

os.makedirs(config['save_dir'], exist_ok=True)
os.makedirs(config['log_dir'], exist_ok=True)

# ========== 分析数据分布 ==========
print("\n📊 分析数据分布...")

train_dataset = FundusDataset(train_dir, excel_dir, is_training=True)
val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

# 收集完整标签统计
def collect_full_labels(dataset, name):
    labels_list = []
    for i in tqdm(range(len(dataset)), desc=f"收集{name}集所有标签"):
        _, labels, _ = dataset[i]
        labels_list.append(labels.numpy())
        if i % 500 == 0:
            gc.collect()
    labels_array = np.vstack(labels_list)
    
    pos_counts = labels_array.sum(axis=0)
    total = len(labels_array)
    
    print(f"\n{name}集完整分布 (共{total}个样本):")
    for i, count in enumerate(pos_counts):
        pct = count / total * 100
        print(f"  类别 {i}: {config['class_names'][i][:10]}: {int(count)} ({pct:.1f}%)")
    
    return labels_array

train_labels = collect_full_labels(train_dataset, "训练")
val_labels = collect_full_labels(val_dataset, "验证")

# ========== 极端过采样策略 ==========
print("\n⚖️ 创建极端过采样策略...")

# 定义各类别的采样权重
class_sample_weights = {
    0: 1.0,   # 正常 (1596样本)
    1: 1.0,   # 糖尿病视网膜病变 (1584样本)
    2: 8.0,   # 青光眼 (304样本)
    3: 8.0,   # 白内障 (296样本)
    4: 15.0,  # 黄斑变性 (232样本)
    5: 20.0,  # 高血压视网膜病变 (146样本)
    6: 8.0,   # 近视 (246样本)
    7: 1.0    # 其他 (1374样本)
}

sample_weights = np.ones(len(train_dataset))

for i in tqdm(range(len(train_dataset)), desc="计算采样权重"):
    _, labels, _ = train_dataset[i]
    
    weight = 1.0
    for class_idx in range(8):
        if labels[class_idx] == 1:
            weight *= class_sample_weights[class_idx]
    
    sample_weights[i] = np.clip(weight, 0.5, 25.0)

print(f"权重范围: {sample_weights.min():.2f} - {sample_weights.max():.2f}")
print(f"权重均值: {sample_weights.mean():.2f}")

# 过采样倍数
oversample_factor = 5
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights) * oversample_factor,
    replacement=True
)

# ========== DataLoader ==========
batch_size = 16
num_workers = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=num_workers,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False
)

print(f"训练批次: {len(train_loader)}")
print(f"验证批次: {len(val_loader)}")

# ========== 加载 Swin Transformer 模型 ==========
print("\n🤖 加载 Swin Transformer 预训练模型...")

# 可选的 Swin Transformer 模型变体
# 'swin_base_patch4_window7_224' - 基础模型
# 'swin_large_patch4_window7_224' - 大模型
# 'swin_small_patch4_window7_224' - 小模型
# 'swin_tiny_patch4_window7_224' - 极小模型
model_name = 'swin_tiny_patch4_window7_224'  # 使用小模型，避免显存不足

model = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=config['num_classes']
)

# 修改分类头，增加Dropout
in_features = model.head.in_features
model.head = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, config['num_classes'])
)

model = model.to(device)
print(f"模型: {model_name}")
print(f"参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# ========== 损失函数 ==========
print("\n⚙️ 配置损失函数和优化器...")

# 为少数类设置更高权重
class_loss_weights = torch.tensor([1.0, 1.0, 6.0, 6.0, 12.0, 15.0, 6.0, 1.0], dtype=torch.float32)
class_loss_weights = class_loss_weights.to(device)

class CombinedLoss(nn.Module):
    def __init__(self, class_weights, alpha=0.5, gamma=2.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        self.focal = FocalLoss(alpha=class_weights, gamma=gamma)
        self.alpha = alpha
    
    def forward(self, inputs, targets):
        return self.alpha * self.bce(inputs, targets) + (1 - self.alpha) * self.focal(inputs, targets)

criterion = CombinedLoss(class_weights=class_loss_weights, alpha=0.5, gamma=2.0)
print("✅ 使用组合损失: 0.5*BCE + 0.5*Focal, gamma=2.0")

# 优化器
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-6,
    weight_decay=0.1
)

# 学习率调度器
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3,
    verbose=True,
    min_lr=1e-7
)

# ========== TensorBoard ==========
writer = SummaryWriter(config['log_dir'])
print(f"TensorBoard日志: {config['log_dir']}")

# ========== 训练参数 ==========
best_val_f1 = 0
best_model_state = None
best_thresholds = None
patience_counter = 0
early_stop_patience = 6
epochs = 25
global_step = 0

print("\n🚀 开始训练 Swin Transformer...")
print(f"总轮数: {epochs}")
print(f"早停patience: {early_stop_patience}")
print("="*60)

# ========== 训练循环 ==========
for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{epochs}")
    print('='*50)
    
    # 训练阶段
    model.train()
    train_loss = 0
    train_steps = 0
    
    train_pbar = tqdm(train_loader, desc='Training')
    for images, labels, _ in train_pbar:
        try:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_steps += 1
            
            writer.add_scalar('Batch/Loss', loss.item(), global_step)
            global_step += 1
            
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
        except Exception as e:
            print(f"训练批次错误: {e}")
            continue
    
    if train_steps == 0:
        print("⚠️ 没有成功训练的批次，请检查数据加载器")
        break
    
    avg_train_loss = train_loss / train_steps
    
    # 验证阶段
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc='Validating'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    
    # 阈值搜索
    best_thresholds = []
    per_class_best_f1 = []
    
    for i in range(8):
        best_f1 = 0
        best_th = 0.5
        for th in np.arange(0.1, 0.9, 0.05):
            preds = (all_probs[:, i] > th).astype(int)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_th = th
        best_thresholds.append(best_th)
        per_class_best_f1.append(best_f1)
    
    # 使用最佳阈值
    val_preds = np.zeros_like(all_probs)
    for i in range(8):
        val_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)
    
    # 计算指标
    val_f1_macro = f1_score(all_labels, val_preds, average='macro', zero_division=0)
    val_f1_micro = f1_score(all_labels, val_preds, average='micro', zero_division=0)
    per_class_actual_f1 = f1_score(all_labels, val_preds, average=None, zero_division=0)
    
    # 少数类F1
    class4_f1 = per_class_actual_f1[4]
    class5_f1 = per_class_actual_f1[5]
    minority_f1 = np.mean([per_class_actual_f1[i] for i in [2, 3, 4, 5, 6]])
    
    print(f"\n📊 验证结果:")
    print(f"  Loss: {avg_train_loss:.4f}")
    print(f"  Macro F1: {val_f1_macro:.4f}")
    print(f"  类别4(黄斑变性) F1: {class4_f1:.4f}")
    print(f"  类别5(高血压) F1: {class5_f1:.4f}")
    print(f"  各类别F1: {[f'{f:.3f}' for f in per_class_actual_f1]}")
    
    # 记录到TensorBoard
    writer.add_scalar('Train/Loss', avg_train_loss, epoch)
    writer.add_scalar('Val/Macro_F1', val_f1_macro, epoch)
    writer.add_scalar('Val/Class4_F1', class4_f1, epoch)
    writer.add_scalar('Val/Class5_F1', class5_f1, epoch)
    writer.add_scalar('Val/Minority_F1', minority_f1, epoch)
    
    # 学习率调度
    scheduler.step(val_f1_macro)
    
    # 保存最佳模型
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_thresholds = best_thresholds.copy()
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': best_model_state,
            'val_f1': val_f1_macro,
            'per_class_f1': per_class_actual_f1,
            'best_thresholds': best_thresholds,
            'config': config,
            'model_name': model_name
        }
        
        torch.save(checkpoint, os.path.join(config['save_dir'], 'best_model_swin.pth'))
        print(f"  ✅ 保存最佳模型! F1={val_f1_macro:.4f}")
    else:
        patience_counter += 1
        print(f"  ⏳ 早停计数: {patience_counter}/{early_stop_patience}")
    
    # 清理内存
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    if patience_counter >= early_stop_patience:
        print(f"\n⏹️ 早停: {early_stop_patience}轮未提升")
        break

# ========== 最终测试 ==========
print("\n" + "="*60)
print("🧪 最终测试")
print("="*60)

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.eval()
    
    test_probs = []
    test_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            test_probs.append(probs)
            test_labels.append(labels.numpy())
    
    test_probs = np.vstack(test_probs)
    test_labels = np.vstack(test_labels)
    
    test_preds = np.zeros_like(test_probs)
    for i in range(8):
        test_preds[:, i] = (test_probs[:, i] > best_thresholds[i]).astype(int)
    
    test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    test_per_class_f1 = f1_score(test_labels, test_preds, average=None, zero_division=0)
    
    print(f"\n📊 最终测试结果:")
    print(f"  Macro F1: {test_f1_macro:.4f}")
    print(f"  类别4(黄斑变性) F1: {test_per_class_f1[4]:.4f}")
    print(f"  类别5(高血压) F1: {test_per_class_f1[5]:.4f}")
    print(f"  各类别F1: {[f'{f:.3f}' for f in test_per_class_f1]}")
    
    # 保存结果
    results_df = pd.DataFrame({
        'Class': config['class_names'],
        'F1_Score': test_per_class_f1,
        'Threshold': best_thresholds
    })
    results_df.to_csv('test_results_swin.csv', index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 训练完成！最佳验证F1: {best_val_f1:.4f}")
    print(f"最终测试F1: {test_f1_macro:.4f}")

writer.close()
print(f"\nTensorBoard: tensorboard --logdir {config['log_dir']}")

✅ 成功导入自定义模块
使用设备: cuda

📊 分析数据分布...
训练集大小: 4906
验证集大小: 1046
测试集大小: 1048


收集训练集所有标签: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4906/4906 [00:40<00:00, 121.72it/s]



训练集完整分布 (共4906个样本):
  类别 0: 正常: 1596 (32.5%)
  类别 1: 糖尿病视网膜病变: 1584 (32.3%)
  类别 2: 青光眼: 304 (6.2%)
  类别 3: 白内障: 296 (6.0%)
  类别 4: 黄斑变性: 232 (4.7%)
  类别 5: 高血压视网膜病变: 146 (3.0%)
  类别 6: 近视: 246 (5.0%)
  类别 7: 其他: 1374 (28.0%)


收集验证集所有标签: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1046/1046 [00:05<00:00, 192.82it/s]



验证集完整分布 (共1046个样本):
  类别 0: 正常: 342 (32.7%)
  类别 1: 糖尿病视网膜病变: 336 (32.1%)
  类别 2: 青光眼: 62 (5.9%)
  类别 3: 白内障: 62 (5.9%)
  类别 4: 黄斑变性: 50 (4.8%)
  类别 5: 高血压视网膜病变: 30 (2.9%)
  类别 6: 近视: 50 (4.8%)
  类别 7: 其他: 288 (27.5%)

⚖️ 创建极端过采样策略...


计算采样权重: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4906/4906 [00:37<00:00, 130.85it/s]


权重范围: 1.00 - 25.00
权重均值: 3.46
训练批次: 1533
验证批次: 66

🤖 加载 Swin Transformer 预训练模型...


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /timm/swin_tiny_patch4_window7_224.ms_in1k/resolve/main/model.safetensors (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000024DF36AFA60>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 97094f87-54ec-4170-80e3-2845977b3ef6)')' thrown while requesting HEAD https://huggingface.co/timm/swin_tiny_patch4_window7_224.ms_in1k/resolve/main/model.safetensors
Retrying in 1s [Retry 1/5].
